# Data Collection Using Web Scraping

The exercise retrieves the Wikipedia Falcon 9 and Falcon Heavy launch-history table with `requests` and `BeautifulSoup`. Website layout can change, so the table selection and column names are inspected before analysis.

In [ ]:
import io, requests, pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
url = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
r = requests.get(url, timeout=45, headers={'User-Agent':'Educational data science capstone/1.0'})
r.raise_for_status()
soup = BeautifulSoup(r.text, 'html.parser')
tables = soup.find_all('table', class_='wikitable')
len(tables)

In [ ]:
# Select a table whose headers resemble a launch list; print headers to verify before proceeding.
selected = None
for table in tables:
    headers = ' '.join(th.get_text(' ', strip=True).lower() for th in table.find_all('th'))
    if 'launch' in headers and ('date' in headers or 'flight' in headers):
        selected = table
        break
if selected is None:
    raise RuntimeError('Could not locate a launch-history table; inspect tables and update the selector.')
scraped = pd.read_html(io.StringIO(str(selected)))[0]
print(scraped.shape)
print(list(scraped.columns))
scraped.head()

In [ ]:
# Store source snapshot. Multi-level/duplicate headers may vary over time; normalize for analysis.
scraped.columns = ['_'.join(map(str, c)).strip() if isinstance(c, tuple) else str(c).strip() for c in scraped.columns]
scraped.to_csv(DATA / 'wikipedia_launch_table.csv', index=False)
print('Saved data/wikipedia_launch_table.csv')